In [61]:
from flax import nnx
import jax
import jax.numpy as jnp
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
import shlex
from collections import defaultdict
sys.path.append("..")

from qnrl.algorithms.algorithm import Algorithm

import wandb
api = wandb.Api()

In [62]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [63]:
project_id = "qnrl"

In [64]:
# List of sweep names and their IDs for reference.
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': 'sgi6zb0t',
    'qnrl/CliffWalking-v1': 'h4g5er22',
    'qnrl/Acrobot-v1': 'sd4va92r',
    'qnrl/CartPole-v1': '4ykq4hvw',
    
    # C51.
    'c51/FrozenLake-v1': '3exa9jvo',
    'c51/CliffWalking-v1': '1nlfftsy',
    'c51/Acrobot-v1': 's3xwn85k',
    'c51/CartPole-v1': '7raha9om',
    
    # Breakout-v5.
    'qnrl/Breakout-v5': 'yavb5d6j',
    'c51/Breakout-v5': 'gxpmvyxv',
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': 'x8921x7e',
    'c51/SpaceInvaders-v5': 'zo5jkyjm',
}

# Best runs tagged for each sweep.
# Retrieve via the code:
#    {sweep_name: [run.id for run in api.sweep(f"{project_id}/{sweep_id}").runs if "best-seed-group" in run.tags] for sweep_name, sweep_id in sweep_name_id.items()}
best_runs_for_sweep = { # name: [id]
    # QnRL.
    "qnrl/FrozenLake-v1": ["asqt9rhf", "pfhzc37a", "tcmzk822", "o8bvmgah"],
    "qnrl/CliffWalking-v1": ["gg17kbpu", "3vn1egt0", "j9n68hn4", "au4el5rn"],
    "qnrl/Acrobot-v1": ["5m2sdda0", "oehcc3xd", "40j6hb1a", "l1c029ff"],
    "qnrl/CartPole-v1": ["atroi3e3", "ridysmu3", "z5elw4vo", "inwzalpf"],
    
    # C51.
    "c51/FrozenLake-v1": ["3dow3k33", "69lp8uef", "ard9oh61", "6ozx435e"],
    "c51/CliffWalking-v1": ["5ti67g7h", "gbkzvg60", "bgrjs1z8", "nk5iyhn3"],
    # "c51/Acrobot-v1": ["5tai0d5b", "57wv5oow", "nqtj0noh", "6xnb70dg"], # max eval/episode_reward, but very large.
    "c51/Acrobot-v1": ["h4xq6mf0", "hlqi1kae", "36eorruy", "6sgyeyvq"], # next-best eval/episode_reward, but size more comparable to QnRL.
    # "c51/CartPole-v1": ["e0x5vmzu", "xxcaueu9", "jit954em", "x0tak5kj"], # max eval/episode_reward, but very large.
    "c51/CartPole-v1": ["7s948sos", "pcq3nv1k", "q3mylk2i", "4eh2yh8z"], # next-best eval/episode_reward, but size more comparable to QnRL.
    
    # Breakout-v5.
    'qnrl/Breakout-v5': ['5lut1vib', 'hv40n10v', '7bvh3pmo', 'dqh0bsum'],
    'c51/Breakout-v5': ['px8cxhah', '6n0rtly1', 'rvgscwo2', '1vod0zn8'],
    
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': ['3m3uhs2j', '7cxftw2s', '6wyv5doy', 'n5ffu05r'],
    # 'qnrl/SpaceInvaders-v5': ['3m3uhs2j', '7cxftw2s', '6wyv5doy'],
    'c51/SpaceInvaders-v5': ['aud35q1l', 'ycc8zvw7', '8t4zs152', 'rgmymygl'],
    # 'c51/SpaceInvaders-v5': ['aud35q1l', '8t4zs152', 'rgmymygl'],
}

In [65]:
import yaml
import ast
from qnrl.serialization import hash_serializable_object
from typing import Any

def get_run_config(config_file: Path):
    with open(str(config_file), 'r') as f:
        config = yaml.safe_load(f)
    return config

def get_run_metadata(metadata_file: Path):
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    return metadata

def get_run_args(metadata_file: Path) -> dict:
    metadata = get_run_metadata(metadata_file)
    l: list[tuple[str, str]] = [tokens.split('=', 1) for tokens in metadata['args'] if '--' in tokens and '=' in tokens]
    d = dict((key.replace('--', '', 1), ast.literal_eval(val)) for key, val in l)
    return d

def get_run_config_dict_from_metadata(metadata_file: Path) -> dict:
    """Compute run config hash from original run program command string."""
    arg_dict = get_run_args(metadata_file)
    return arg_dict['config']
    
    
    # metadata = get_run_metadata(metadata_file)
    
    # # Get config string from run program command arguments.
    # # Format is: --config="{...}"
    # cfg_str = metadata['args'][1].split("--config=")[1]
    # # Convert this string to dictionary via literal evaluation.
    # cfg_dict = ast.literal_eval(cfg_str)
    # return cfg_dict

In [66]:
from typing import Iterator

def eval(sweep_id: str, run_id: str, eval_kwargs_override: dict = {}) -> Iterator[dict]:
    
    dir_run = root_sweeps/f"sweep-{sweep_id}"/f"run-{run_id}"
    metadata_file = dir_run/"wandb-metadata.json"

    arg_dict = get_run_args(metadata_file)
    
    print(f"Evaluating: sweep={sweep_id}, run={run_id}, num_eval={arg_dict['num_eval']}")

    cfg = arg_dict['config']
    # cfg_hash, _ = hash_serializable_object(cfg) # Compute hash for config object.
    algo_config = Algorithm.from_dict(cfg)
    
    # Load parameters from model save.
    model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
    algo_config.load(model_dirs)
    # nnx.display(nnx.state(algo_config.model))
    
    eval_reports = []
    eval_seed = arg_dict['eval_seed'] if arg_dict['eval_seed'] is not None else algo_config.seed
    key = jax.random.PRNGKey(seed=eval_seed)
    for eval_idx in range(arg_dict['num_eval']):
        
        # Generate a new random key.
        key, subkey = jax.random.split(key=key)
        
        # Evaluate the model using a different random seed.
        seed_eval = int(subkey[0])
        eval_kwargs = {}
        if 'eval_total_timesteps' in arg_dict and arg_dict['eval_total_timesteps'] is not None:
            eval_kwargs['total_timesteps'] = arg_dict['eval_total_timesteps']
        _, eval_report = algo_config.eval(seed=seed_eval, **{**eval_kwargs, **eval_kwargs_override})
        
        # Add an index counter for the evaluation report.
        eval_report['eval/index'] = eval_idx
        
        yield eval_report


def eval_via_env(env_key: str, eval_kwargs_override: dict = {}):
    eval_reports = defaultdict(lambda: defaultdict(list[dict])) # { sweep_id: { run_id: [report_0, report_1, ...] } }
    sweep_id = sweep_name_id[env_key]
    for run_id in best_runs_for_sweep[env_key]:
        eval_reports[sweep_id][run_id] = list(eval(sweep_id, run_id, eval_kwargs_override))
    return eval_reports

In [67]:
# env_key = "qnrl/CartPole-v1"
# env_key = "c51/CartPole-v1"

# env_key = "qnrl/FrozenLake-v1"
# env_key = "c51/FrozenLake-v1"

# env_key = "qnrl/Acrobot-v1"
# env_key = "c51/Acrobot-v1"

env_key = "qnrl/SpaceInvaders-v5"
# env_key = "c51/SpaceInvaders-v5"

# eps = 0.05 # What Bellemare used for training.
eps = 0.001 # What Bellemare used for eval.
report_dict = eval_via_env(env_key, eval_kwargs_override=dict(eps=eps))

Evaluating: sweep=x8921x7e, run=3m3uhs2j, num_eval=10


 19%|█▊        | 187/1000 [00:11<00:50, 16.07step/s, eval/step=187, eval/episode_reward=125, eval/episode_length=188, eval/episode_time=11.6]


Evaluating: sweep=x8921x7e, run=7cxftw2s, num_eval=10


 59%|█████▉    | 590/1000 [00:36<00:25, 16.35step/s, eval/step=590, eval/episode_reward=425, eval/episode_length=591, eval/episode_time=36]


Evaluating: sweep=x8921x7e, run=6wyv5doy, num_eval=10


 30%|███       | 301/1000 [00:18<00:42, 16.40step/s, eval/step=301, eval/episode_reward=120, eval/episode_length=302, eval/episode_time=18.3]


Evaluating: sweep=x8921x7e, run=n5ffu05r, num_eval=10


 19%|█▉        | 188/1000 [00:11<00:50, 16.01step/s, eval/step=188, eval/episode_reward=110, eval/episode_length=189, eval/episode_time=11.7]


In [68]:
nnx.display(report_dict)

In [69]:
def get_metric_as_series_for_run(metric_key: str, run_id: str, eval_reports: list[dict]):
    """Extract metric key as Pandas series and change metric key to row ID."""
    return pd.DataFrame(eval_reports)[metric_key].rename(run_id)

def get_metric_as_dataframe_for_sweep(metric_key: str, sweep_id: str):
    l = []
    for run_id, eval_reports in report_dict[sweep_id].items():
        # l.append(pd.DataFrame(eval_reports)['eval/episode_reward'].to_list())
        l.append(pd.DataFrame(eval_reports)[metric_key].rename(run_id))
    df = pd.concat(l, axis=1, join='outer') # column keys are 'row_id'.
    return df

In [70]:
for sweep_id in report_dict.keys():
    df = get_metric_as_dataframe_for_sweep('eval/episode_reward', sweep_id)
    print(df)
    print(df.mean())
    print(df.mean().mean())

   3m3uhs2j  7cxftw2s  6wyv5doy  n5ffu05r
0      60.0      30.0     150.0      35.0
1     140.0     155.0      65.0     115.0
2     140.0     155.0      65.0     115.0
3     125.0     425.0     120.0     110.0
4      10.0     120.0     150.0      35.0
5     125.0     425.0     120.0     110.0
6     125.0     335.0     120.0     110.0
7     125.0     425.0     120.0     110.0
8     125.0     425.0     120.0     110.0
9     125.0     425.0     120.0     110.0
3m3uhs2j    110.0
7cxftw2s    292.0
6wyv5doy    115.0
n5ffu05r     96.0
dtype: float64
153.25
